# Check the clisops subset operator against CMIP6 QA/QC findings

Version 2026-11-13

The final table covers all 14 QA/QC findings. Synthetic tests check xarray decoding and combined time and geographic bounding-box (bbox) subsetting for the findings below.

| Finding | Metadata issue |
| --- | --- |
| [3.1] | Unpadded or hyphen-separated reference time in the time-units string |
| [3.2] | Shifted timestamps |
| [3.3] | Missing coordinate `axis` or `standard_name` |
| [3.7] | Projected x/y axes with degree units |
| [3.8] | Area variable without a geographic coordinate reference |
| [3.12] | Reference to an absent auxiliary `height` coordinate |

Candidate model / experiment / variant combinations are partial ESGF identifiers for future real-data validation.

In [1]:
import re
import sys
from contextlib import redirect_stdout
from importlib.metadata import version
from io import StringIO

import ipytest
import numpy as np
import pytest
import xarray as xr
from IPython.display import Markdown, display
from clisops.ops.subset import subset
from netCDF4 import Dataset

ipytest.autoconfig(raise_on_error=True, magics=False, addopts=())

print(f"Python {sys.version.split()[0]}")
for package in ("clisops", "xarray", "cf-xarray", "numpy", "netCDF4", "cftime", "pytest", "ipytest"):
    print(f"{package} {version(package)}")

Python 3.13.18
clisops 0.18.1
xarray 2025.10.1
cf-xarray 0.11.3
numpy 2.5.4
netCDF4 1.7.5
cftime 1.6.5
pytest 9.2.0
ipytest 0.14.2


## Shared test data

Select from 2 January 1850 at 00:00 through 3 January at 00:00, inclusive. The bbox spans 12°W to 3°W and 43°N to 52°N.

In [2]:
TIME_UNITS = "days since 1850-01-01 00:00:00"
DATES = np.arange("1850-01-01", "1850-01-05", dtype="datetime64[D]").astype("datetime64[ns]")
TIME_RANGE = "1850-01-02T00:00:00/1850-01-03T00:00:00"
BBOX = (-12, 43, -3, 52)  # west, south, east, north

time = (
    "time",
    np.arange(4),
    {"units": TIME_UNITS, "calendar": "standard", "axis": "T", "standard_name": "time"},
)
latitude = (
    "lat",
    [40, 45, 50, 55],
    {"units": "degrees_north", "axis": "Y", "standard_name": "latitude"},
)
longitude = (
    "lon",
    [-15, -10, -5, 0, 5],
    {"units": "degrees_east", "axis": "X", "standard_name": "longitude"},
)
pressure = (
    "plev",
    [100000, 85000],
    {"units": "Pa", "axis": "Z", "standard_name": "air_pressure", "positive": "down"},
)

rectilinear = xr.Dataset(
    {"ta": (("time", "plev", "lat", "lon"), np.arange(160.0).reshape(4, 2, 4, 5))},
    coords={"time": time, "plev": pressure, "lat": latitude, "lon": longitude},
)
rectilinear_bbox = {"lat": slice(1, 3), "lon": slice(1, 3)}

In [3]:
FINDINGS = {
    "3.1": "Time-unit string formatting",
    "3.2": "Temporal coordinate positioning",
    "3.3": "Coordinate axis / standard_name metadata",
    "3.4": "Parent-simulation controlled vocabulary",
    "3.5": "Physical-variable standard_name mismatches",
    "3.6": "Equivalent numeric notation for sos units",
    "3.7": "Horizontal x/y coordinate units",
    "3.8": "Geographic referencing of area",
    "3.9": "Recommended metadata on bnds",
    "3.10": "Ensemble-member index inconsistencies",
    "3.11": "Global realization_index attribute type",
    "3.12": "Missing auxiliary height coordinate",
    "3.13": "Time and spatial bounds inconsistencies",
    "3.14": "Nominal-resolution vocabulary",
}

cases = []
shifted_time_cases = []


def add_case(finding, name, raw, variable, bbox_index, bbox_mask=None, *, dates=DATES, offset_hours=None):
    # Derive the expected selection independently from the fixture dates and grid indices.
    start, end = np.array(TIME_RANGE.split("/"), dtype="datetime64[ns]")
    expected = raw[variable].assign_coords(time=dates)
    expected_subset = expected.isel(time=(start <= dates) & (dates <= end), **bbox_index)
    if bbox_mask is not None:
        expected_subset = expected_subset.where(bbox_mask)
    case = (raw, expected, expected_subset)
    case_id = f"{finding}-{name}"
    if offset_hours is None:
        cases.append(pytest.param(case, id=case_id))
    else:
        cases.append(pytest.param(case, id=f"{case_id}-unfixed"))
        shifted_time_cases.append(pytest.param(case, offset_hours, id=f"{case_id}-fixed"))

## Finding [3.1] — Time-unit string formatting

The fixtures reproduce `days since 1850-01-01 0:0:0.0` and `days since 1850-01-01-00-00-00` as written. The control uses padded time fields. Pytest checks the decoded dates and subset.

Candidate for future real-data validation — `CanESM5` / `historical` / `r1i1p1f1`.

In [4]:
add_case("3.1+3.2+3.3", "rectilinear-control", rectilinear, "ta", rectilinear_bbox)

short_time_units = rectilinear.copy(deep=True)
short_time_units.time.attrs["units"] = "days since 1850-01-01 0:0:0.0"
add_case("3.1", "unpadded-time-units", short_time_units, "ta", rectilinear_bbox)

hyphenated_time_units = rectilinear.copy(deep=True)
hyphenated_time_units.time.attrs["units"] = "days since 1850-01-01-00-00-00"
add_case("3.1", "hyphenated-time-units", hyphenated_time_units, "ta", rectilinear_bbox)

## Finding [3.2] — Temporal coordinate positioning

Shift the timestamps later by 12 hours or 1 hour (MEDIUM, QA/QC artefacts) or by 1 day (HIGH). The data values, request, and bbox are shared with the unshifted control. The baseline checks which records fall within the request according to the timestamps stored in each fixture.

A notebook-local Woodpecker-style fix subtracts the known fixture offset from the decoded time coordinate before calling clisops with the same request. The repair tests compare the full corrected dataset and the resulting subset with the unshifted control. A production Woodpecker rule would need a verified dataset-specific offset. These fixtures have no time bounds.

In [5]:
def repair_time_offset(dataset, offset_hours):
    corrected_time = dataset.time.copy(data=dataset.time.values - np.timedelta64(offset_hours, "h"))
    return dataset.assign_coords(time=corrected_time)


SHIFT_VARIANTS = (
    ("12-hour-shift", "12-hour shift", 12, "MEDIUM"),
    ("1-hour-shift", "1-hour shift", 1, "MEDIUM"),
    ("1-day-shift", "1-day shift", 24, "HIGH"),
)

for label, _, hours, _ in SHIFT_VARIANTS:
    shifted = rectilinear.assign_coords(
        time=("time", rectilinear.time.values + hours / 24, rectilinear.time.attrs),
    )
    add_case(
        "3.2", label, shifted, "ta", rectilinear_bbox,
        dates=DATES + np.timedelta64(hours, "h"), offset_hours=hours,
    )

## Finding [3.3] — Coordinate axis / standard_name metadata

The variants probe missing `axis` or `standard_name` on `time`, `lat`, `lon`, or `plev`, one attribute at a time. The expected subset retains both pressure levels.

Candidate for future real-data validation — `CNRM-CM6-1` / `historical` / `r1i1p1f2`.

In [6]:
for coordinate in ("time", "lat", "lon", "plev"):
    for attribute in ("axis", "standard_name"):
        missing_attribute = rectilinear.copy(deep=True)
        del missing_attribute[coordinate].attrs[attribute]
        add_case("3.3", f"{coordinate}-without-{attribute}", missing_attribute, "ta", rectilinear_bbox)

## Finding [3.7] — Horizontal x/y coordinate units

The variant uses degree-valued projected x/y axes and retains valid 2-D latitude and longitude. The expected bbox keeps a 2 × 2 window and masks one data cell. The negative control checks the missing-longitude error when both geographic coordinates are removed.

Candidate for future real-data validation — `MIROC6` / `historical` / `r1i1p1f1`.

In [7]:
curvilinear = xr.Dataset(
    {"tas": (("time", "y", "x"), np.arange(48.0).reshape(4, 3, 4))},
    coords={
        "time": time,
        "x": ("x", [0, 500_000, 1_000_000, 1_500_000],
              {"standard_name": "projection_x_coordinate", "units": "m", "axis": "X"}),
        "y": ("y", [0, 500_000, 1_000_000],
              {"standard_name": "projection_y_coordinate", "units": "m", "axis": "Y"}),
        "lat": (("y", "x"), [[39, 40, 41, 42], [43, 45, 46, 47], [49, 50, 53, 54]],
                {"standard_name": "latitude", "units": "degrees_north"}),
        "lon": (("y", "x"), [[-16, -11, -6, -1], [-15, -10, -5, 0], [-14, -9, -4, 1]],
                {"standard_name": "longitude", "units": "degrees_east"}),
    },
)
curvilinear_bbox = {"y": slice(1, 3), "x": slice(1, 3)}
curvilinear_mask = xr.DataArray([[True, True], [True, False]], dims=("y", "x"))

add_case("3.7+3.8", "projected-axes-in-metres", curvilinear, "tas", curvilinear_bbox, curvilinear_mask)

axes_in_degrees = curvilinear.assign_coords(
    x=("x", [0, 5, 10, 15], {**curvilinear.x.attrs, "units": "degrees"}),
    y=("y", [0, 5, 10], {**curvilinear.y.attrs, "units": "degrees"}),
)
add_case("3.7", "projected-axes-in-degrees", axes_in_degrees, "tas", curvilinear_bbox, curvilinear_mask)

## Finding [3.8] — Geographic referencing of area

The fixture omits the geographic coordinate reference from `area` and keeps it on `tas`. The test checks both subsets, including the `area` mask and geographic coordinates.

In [8]:
area_without_reference = curvilinear.assign(
    area=(("y", "x"), 1e6 + np.arange(12.0).reshape(3, 4), {"units": "m2"}),
)
area_without_reference.area.encoding["coordinates"] = None
expected_area = area_without_reference.area.isel(curvilinear_bbox).where(curvilinear_mask)
add_case("3.8", "area-without-coordinate-reference",
         area_without_reference, "tas", curvilinear_bbox, curvilinear_mask)

## Finding [3.12] — Missing auxiliary height coordinate

The variant reproduces a dangling `coordinates="height"` reference on `tas`. The fixture verifies the reference on disk and after xarray decoding.

Candidate for future real-data validation — `KIOST-ESM` / `historical` / `r1i1p1f1`.

In [9]:
height_control = xr.Dataset(
    {"tas": (("time", "lat", "lon"), np.arange(80.0).reshape(4, 4, 5), {"coordinates": "height"})},
    coords={
        "time": time,
        "lat": latitude,
        "lon": longitude,
        "height": ((), 2.0, {"standard_name": "height", "units": "m", "positive": "up"}),
    },
)
add_case("3.12", "height-control", height_control, "tas", rectilinear_bbox)
missing_height = height_control.drop_vars("height")
add_case("3.12", "missing-height", missing_height, "tas", rectilinear_bbox)

## Pytest tests

The fixture checks the NetCDF metadata and decoded coordinates before subsetting. The repair tests reopen the same shifted fixtures and apply the correction after decoding.

In [10]:
ipytest.clean()


@pytest.fixture(params=cases)
def decoded_case(request, tmp_path):
    raw, expected, expected_subset = request.param
    path = tmp_path / "fixture.nc"
    raw.to_netcdf(path, engine="netcdf4")

    with Dataset(path) as file:
        assert file["time"].units == raw["time"].attrs["units"]
        if raw is area_without_reference:
            assert {"lat", "lon"} <= set(file["tas"].coordinates.split())
            assert "coordinates" not in file["area"].ncattrs()
        if raw is height_control or raw is missing_height:
            assert file[expected.name].coordinates == "height"
            assert ("height" in file.variables) == (raw is height_control)

    with xr.open_dataset(path, engine="netcdf4") as dataset:
        decoded = dataset.load()

    assert decoded[expected.name].equals(expected)
    for name, coordinate in raw.coords.items():
        for attr in ("axis", "standard_name", "units"):
            actual = decoded[name].encoding.get(attr, decoded[name].attrs.get(attr))
            assert actual == coordinate.attrs.get(attr), f"{name}.{attr}: {actual!r}"
    if raw is area_without_reference:
        assert decoded.area.encoding.get("coordinates") is None
    if raw is missing_height:
        assert decoded[expected.name].encoding.get("coordinates") == "height"
        assert "height" not in decoded.variables

    return decoded, expected_subset


def print_time_samples(actual):
    timestamps = np.datetime_as_string(actual.time.values, unit="m")
    values = actual.isel(plev=0, lat=0, lon=0).values
    pairs = ", ".join(f"{stamp} = {value:g}" for stamp, value in zip(timestamps, values))
    print(f"Returned timestamps and sample values  {pairs}")


def test_time_and_bbox_subset(decoded_case, request):
    decoded, expected_subset = decoded_case
    outputs = subset(decoded, time=TIME_RANGE, area=BBOX, output_type="xarray")
    assert len(outputs) == 1
    actual = outputs[0][expected_subset.name]
    if not actual.time.equals(expected_subset.time):
        expected_times = ", ".join(np.datetime_as_string(expected_subset.time.values, unit="s"))
        returned_times = ", ".join(np.datetime_as_string(actual.time.values, unit="s"))
        pytest.fail(
            f"Requested interval  {TIME_RANGE}\n"
            f"Expected timesteps  {expected_times}\n"
            f"Returned timesteps  {returned_times}",
        )
    assert actual.equals(expected_subset), "Subset coordinates or values differ from the expected selection"

    findings = request.node.callspec.id.split("-", 1)[0].split("+")
    if "3.2" in findings:
        print_time_samples(actual)
    if "area" in decoded.data_vars:
        assert outputs[0].area.equals(expected_area)


@pytest.mark.parametrize(("decoded_case", "offset_hours"), shifted_time_cases, indirect=["decoded_case"])
def test_time_and_bbox_subset_after_repair(decoded_case, offset_hours):
    decoded, _ = decoded_case
    repaired = repair_time_offset(decoded, offset_hours)
    control = xr.decode_cf(rectilinear)
    xr.testing.assert_identical(repaired, control)

    expected_subset = control.ta.isel(time=slice(1, 3), **rectilinear_bbox)
    (output,) = subset(repaired, time=TIME_RANGE, area=BBOX, output_type="xarray")
    xr.testing.assert_equal(output.ta, expected_subset)
    print_time_samples(output.ta)


@pytest.mark.parametrize("source", [pytest.param(axes_in_degrees, id="3.7-missing-geographic-coordinates")])
def test_bbox_without_geographic_coordinates(source):
    without_lat_lon = xr.decode_cf(source.drop_vars(["lat", "lon"]))
    with pytest.raises(AttributeError, match="has no attribute 'lon'") as error:
        subset(without_lat_lon, area=BBOX, output_type="xarray")

    print(f"Bbox without geographic coordinates raised {error.type.__name__}: {error.value}")

## Run pytest

The fixed and unfixed cases for all three [3.2] variants run in the same session as the unshifted control and the other findings.

In [11]:
class FindingResults:
    def __init__(self):
        self.tests = {finding: [] for finding in FINDINGS}
        self.outcomes = {}

    def pytest_collection_finish(self, session):
        for item in session.items:
            case_id = item.callspec.id
            if case_id.startswith("3.2-"):
                self.tests.setdefault(case_id, []).append(item.nodeid)
                continue
            # A shared control can contribute to more than one finding.
            for finding in case_id.split("-", 1)[0].split("+"):
                self.tests[finding].append(item.nodeid)

    def pytest_runtest_logreport(self, report):
        phase_outcomes = self.outcomes.setdefault(report.nodeid, {})
        phase_outcomes[report.when] = "skipped" if hasattr(report, "wasxfail") else report.outcome

    def status(self, tests):
        if not tests:
            return "Not run"
        outcomes = [self.outcomes.get(nodeid, {}) for nodeid in tests]
        passed = {"setup": "passed", "call": "passed", "teardown": "passed"}
        if any("failed" in phases.values() for phases in outcomes):
            return "FAIL"
        if outcomes and all(phases == passed for phases in outcomes):
            return "PASS"
        return "INCOMPLETE"


results = FindingResults()
with redirect_stdout(StringIO()) as report:
    ipytest.run(
        "-v", "-rP", "--tb=long", "--no-header", "--color=yes", "-p", "no:cacheprovider",
        plugins=[results], raise_on_error=False,
    )

# Replace generated module names and kernel paths with readable notebook locations.
output = re.sub(r"t_[0-9a-f]{32}\.py::", "", report.getvalue())
output = re.sub(r"t_[0-9a-f]{32}\.py(?=:)", "Notebook tests", output)
output = re.sub(
    r"^((?:\x1b\[[0-9;]*m)*).*[/\\]ipykernel_\d+[/\\]\d+\.py(?=(?:\x1b\[[0-9;]*m)*:\d+)",
    r"\1Notebook cell", output, flags=re.MULTILINE,
)
print(output, end="")

======================================= test session starts ========================================
collecting ... collected 23 items

test_time_and_bbox_subset[3.1+3.2+3.3-rectilinear-control] PASSED [  4%]
test_time_and_bbox_subset[3.1-unpadded-time-units] PASSED [  8%]
test_time_and_bbox_subset[3.1-hyphenated-time-units] PASSED [ 13%]
test_time_and_bbox_subset[3.2-12-hour-shift-unfixed] FAILED [ 17%]
test_time_and_bbox_subset[3.2-1-hour-shift-unfixed] FAILED [ 21%]
test_time_and_bbox_subset[3.2-1-day-shift-unfixed] PASSED [ 26%]
test_time_and_bbox_subset[3.3-time-without-axis] PASSED [ 30%]
test_time_and_bbox_subset[3.3-time-without-standard_name] PASSED [ 34%]
test_time_and_bbox_subset[3.3-lat-without-axis] PASSED [ 39%]
test_time_and_bbox_subset[3.3-lat-without-standard_name] PASSED [ 43%]
test_time_and_bbox_subset[3.3-lon-without-axis] PASSED [ 47%]
test_time_and_bbox_subset[3.3-lon-without-standard_name] PASSED [ 52%]
test_time_and_bbox_subset[3.3-plev-without-axis] PASSED [ 56

## Results and limitations

QA/QC severity classifies findings for data selection. The Subset column reports the clisops test outcomes for the unfixed fixtures. Each [3.2] row shows the result for that variant, with the notebook-local repair outcome in the Woodpecker column. The unshifted [3.2] control appears in the pytest output. Finding [3.13] is outside this assessment.

In clisops 0.18.1, a requested endpoint absent from the time index can fall back to the dataset time range. The 1- and 12-hour shifts reproduce the resulting out-of-window selections. The 1-day shift passes the baseline check against the recorded dates, but those dates now point to different samples than in the unshifted control. The repair tests check that all three corrected fixtures return the control's timestamps and data values.

The fixtures omit longitude cell bounds, which can trigger warnings. Real CMIP6 datasets and production Woodpecker rules remain unverified.

In [12]:
severity_overrides = {
    "3.13": "HIGH",
}
low_relevance = {"3.4", "3.6", "3.9", "3.10", "3.11", "3.14"}

table = [
    "| Finding | QA/QC severity | Subset | Woodpecker |",
    "| --- | --- | --- | --- |",
]
for finding, description in FINDINGS.items():
    if finding == "3.2":
        for label, shift, _, severity in SHIFT_VARIANTS:
            unfixed = results.status(results.tests.get(f"3.2-{label}-unfixed", []))
            fixed = results.status(results.tests.get(f"3.2-{label}-fixed", []))
            if label == "1-day-shift" and unfixed == "PASS":
                unfixed = "PASS (recorded dates)"
            table.append(f"| [3.2] {shift} | {severity} | {unfixed} | {fixed} after repair |")
        continue

    severity = severity_overrides.get(finding, "MEDIUM")
    status = results.status(results.tests[finding])
    if status == "PASS":
        woodpecker = "Not needed"
    elif status == "Not run":
        woodpecker = "Unlikely needed" if finding in low_relevance else "Undetermined"
    else:
        woodpecker = "To investigate"
    table.append(f"| [{finding}] {description} | {severity} | {status} | {woodpecker} |")

display(Markdown("\n".join(table)))

| Finding | QA/QC severity | Subset | Woodpecker |
| --- | --- | --- | --- |
| [3.1] Time-unit string formatting | MEDIUM | PASS | Not needed |
| [3.2] 12-hour shift | MEDIUM | FAIL | PASS after repair |
| [3.2] 1-hour shift | MEDIUM | FAIL | PASS after repair |
| [3.2] 1-day shift | HIGH | PASS (recorded dates) | PASS after repair |
| [3.3] Coordinate axis / standard_name metadata | MEDIUM | PASS | Not needed |
| [3.4] Parent-simulation controlled vocabulary | MEDIUM | Not run | Unlikely needed |
| [3.5] Physical-variable standard_name mismatches | MEDIUM | Not run | Undetermined |
| [3.6] Equivalent numeric notation for sos units | MEDIUM | Not run | Unlikely needed |
| [3.7] Horizontal x/y coordinate units | MEDIUM | PASS | Not needed |
| [3.8] Geographic referencing of area | MEDIUM | PASS | Not needed |
| [3.9] Recommended metadata on bnds | MEDIUM | Not run | Unlikely needed |
| [3.10] Ensemble-member index inconsistencies | MEDIUM | Not run | Unlikely needed |
| [3.11] Global realization_index attribute type | MEDIUM | Not run | Unlikely needed |
| [3.12] Missing auxiliary height coordinate | MEDIUM | PASS | Not needed |
| [3.13] Time and spatial bounds inconsistencies | HIGH | Not run | Undetermined |
| [3.14] Nominal-resolution vocabulary | MEDIUM | Not run | Unlikely needed |